In [2]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
# Membuat SparkSession
spark = SparkSession.builder \
.appName("HRDataAnalysis") \
.master("local[*]") \
.config("spark.driver.port", "7077") \
.getOrCreate()
print("Spark Version:", spark.version)

Spark Version: 3.5.6


In [3]:
# Membaca dataset CSV
df = spark.read.csv("HRDataAnalysis.csv", header=True,
inferSchema=True)
# Menampilkan 5 data teratas
df.show(5)
# Melihat skema DataFrame
df.printSchema()

+----------+-----------+---------------+----------+--------------------+----------+--------------------+------------------+----------------+--------+---------+----------+
|Unnamed: 0|Employee_ID|      Full_Name|Department|           Job_Title| Hire_Date|            Location|Performance_Rating|Experience_Years|  Status|Work_Mode|Salary_INR|
+----------+-----------+---------------+----------+--------------------+----------+--------------------+------------------+----------------+--------+---------+----------+
|         0| EMP0000001|  Joshua Nguyen|        IT|   Software Engineer|2011-08-10|  Isaacland, Denmark|                 5|              14|Resigned|  On-site|   1585363|
|         1| EMP0000002| Julie Williams| Marketing|      SEO Specialist|2018-03-02|Anthonyside, Cost...|                 2|               7|  Active|  On-site|    847686|
|         2| EMP0000003|Alyssa Martinez|        HR|          HR Manager|2023-03-20|Port Christinapor...|                 1|               2|  Act

In [5]:
# Mengonversi DataFrame ke RDD
rdd = df.rdd
# Transformasi: map - mengambil elemen
print("\na) Transformasi MAP - mengambil gaji dan merubah menjadi format ribuan")
salary_rdd = rdd.map(lambda row: (row['Full_Name'], row['Salary_INR'] / 1000))
# Tampilkan hasil
salary_samples = salary_rdd.take(5)
print("Hasil transformasi MAP (5 sample):")
for name, salary_in_k in salary_samples:
    print(f" {name}: Rp. {salary_in_k}")


a) Transformasi MAP - mengambil gaji dan merubah menjadi format ribuan
Hasil transformasi MAP (5 sample):
 Joshua Nguyen: Rp. 1585.363
 Julie Williams: Rp. 847.686
 Alyssa Martinez: Rp. 1430.084
 Nicholas Valdez: Rp. 990.689
 Joel Hendricks: Rp. 535.082


In [6]:
# Mengonversi DataFrame ke RDD
rdd = df.rdd
# Transformasi: filter - menyaring data
print("b) Transformasi FILTER - Menyaring karyawan dengan gaji 1 juta")
filtered_rdd = rdd.filter(lambda row: row['Salary_INR'] >1000000)
# Menampilkan Hasil
print("10 data pertama hasil filtering (menggunakan take()):")
first_10 = filtered_rdd.take(10)
for i, employee in enumerate(first_10):print(f"{i+1}. {employee}")

b) Transformasi FILTER - Menyaring karyawan dengan gaji 1 juta
10 data pertama hasil filtering (menggunakan take()):
1. Row(Unnamed: 0=0, Employee_ID='EMP0000001', Full_Name='Joshua Nguyen', Department='IT', Job_Title='Software Engineer', Hire_Date=datetime.date(2011, 8, 10), Location='Isaacland, Denmark', Performance_Rating=5, Experience_Years=14, Status='Resigned', Work_Mode='On-site', Salary_INR=1585363)
2. Row(Unnamed: 0=2, Employee_ID='EMP0000003', Full_Name='Alyssa Martinez', Department='HR', Job_Title='HR Manager', Hire_Date=datetime.date(2023, 3, 20), Location='Port Christinaport, Saudi Arabia', Performance_Rating=1, Experience_Years=2, Status='Active', Work_Mode='On-site', Salary_INR=1430084)
3. Row(Unnamed: 0=6, Employee_ID='EMP0000007', Full_Name='Julie Wright', Department='Finance', Job_Title='Finance Manager', Hire_Date=datetime.date(2016, 4, 4), Location='Karenfort, South Africa', Performance_Rating=2, Experience_Years=9, Status='Active', Work_Mode='On-site', Salary_INR=1

In [8]:
# Mengonversi DataFrame ke RDD
rdd = df.rdd
print("c) Transformasi FlatMap - Memecah Full_Name menjadi kata-words individual:")
name_words_rdd = rdd.flatMap(lambda row: row['Full_Name'].split())
print("Kata-words dalam nama (10 pertama):")
name_words_rdd.take(10)

c) Transformasi FlatMap - Memecah Full_Name menjadi kata-words individual:
Kata-words dalam nama (10 pertama):


['Joshua',
 'Nguyen',
 'Julie',
 'Williams',
 'Alyssa',
 'Martinez',
 'Nicholas',
 'Valdez',
 'Joel',
 'Hendricks']

In [7]:
# Mengonversi DataFrame ke RDD
rdd = df.rdd
# Transformasi: reduceByKey - mengurangi berdasarkan key
print("d) Transformasi REDUCEBYKEY - Menghitung total gaji per departemen")
# Pastikan ada kolom 'departemen'
dept_salary_rdd = rdd.map(lambda row: (row['Department'],
row['Salary_INR']))
total_dept_salary = dept_salary_rdd.reduceByKey(lambda a, b:
a + b)
# Menampilkan sample data (misalnya 5 departemen pertama)
print("Sample 5 departemen pertama:")
sample_results = total_dept_salary.take(5)
for dept, total_salary in sample_results:print(f" {dept}: Rp.{total_salary:,.2f}")

d) Transformasi REDUCEBYKEY - Menghitung total gaji per departemen
Sample 5 departemen pertama:
 Finance: Rp.187,962,916,267.00
 Sales: Rp.317,207,725,524.00
 R&D: Rp.79,844,824,817.00
 IT: Rp.679,092,203,055.00
 HR: Rp.118,361,234,839.00


In [7]:
# Mengonversi DataFrame ke RDD
rdd = df.rdd
# Transformasi: reduceByKey - Menampilkan Data Berbeda berdasarkan key
print("e) Transformasi Distinct - Menampilkan Data yang berbeda disetiap baris key")
job_title = rdd.map(lambda x: x.Job_Title).distinct()
print("Job Title unik:", job_title.collect())

e) Transformasi Distinct - Menampilkan Data yang berbeda disetiap baris key
Job Title unik: ['Logistics Coordinator', 'Marketing Executive', 'Supply Chain Manager', 'CFO', 'Account Manager', 'Operations Director', 'Brand Manager', 'Innovation Manager', 'Talent Acquisition Specialist', 'CTO', 'Finance Manager', 'HR Manager', 'HR Executive', 'Research Scientist', 'Data Analyst', 'HR Director', 'SEO Specialist', 'Operations Executive', 'Sales Director', 'DevOps Engineer', 'Lab Technician', 'Software Engineer', 'Sales Executive', 'Content Strategist', 'Product Developer', 'Financial Analyst', 'Accountant', 'IT Manager', 'Business Development Manager']


In [9]:
# Repartisi RDD untuk distribusi yang lebih baik
rdd = df.rdd.repartition(4) # Sesuaikan dengan jumlah core
print(f"Jumlah partisi RDD: {rdd.getNumPartitions()}")
# join → contoh join Employee_ID dengan Status dan Departemen
id_status = rdd.map(lambda x: (x.Employee_ID, x.Status))
id_dept = rdd.map(lambda x: (x.Employee_ID, x.Department))
joined = id_status.join(id_dept)
print("Sample hasil join (10 data pertama):")
sample_results = joined.take(10)
for result in sample_results:print(f" {result}")

Jumlah partisi RDD: 4
Sample hasil join (10 data pertama):
 ('EMP0000031', ('Active', 'IT'))
 ('EMP0000034', ('Terminated', 'IT'))
 ('EMP0000073', ('Active', 'IT'))
 ('EMP0000080', ('Active', 'Operations'))
 ('EMP0000115', ('Active', 'IT'))
 ('EMP0000193', ('Active', 'Marketing'))
 ('EMP0000198', ('Active', 'IT'))
 ('EMP0000199', ('Resigned', 'IT'))
 ('EMP0000237', ('Active', 'HR'))
 ('EMP0000276', ('Active', 'IT'))


In [10]:
# 1. Bagaimana distribusi “Work_Mode” (On-site, Remote)?
work_mode_counts = rdd.map(lambda x: x['Work_Mode']).countByValue()
print("Distribusi Work_Mode:")
for mode, count in work_mode_counts.items():print(f"{mode}: {count}")

Distribusi Work_Mode:
On-site: 1199109
Remote: 800891


In [11]:
# 2. Berapa jumlah karyawan di setiap departemen?
dept_counts = rdd.map(lambda x: x['Department']).countByValue()
print("Jumlah karyawan di setiap departemen:")
for dept, count in dept_counts.items():print(f"{dept}: {count}")

Jumlah karyawan di setiap departemen:
IT: 601042
Sales: 400031
Marketing: 240081
HR: 159119
Finance: 199873
R&D: 99759
Operations: 300095


In [12]:
# 3. Berapa rata-rata gaji per Departemen?
dept_salary_count = rdd.map(lambda x: (x['Department'], (x['Salary_INR'], 1)))
sum_count = dept_salary_count.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
average_salary_per_dept = sum_count.mapValues(lambda x: x[0] / x[1])
print("Rata-rata gaji per departemen:")
for dept, avg_salary in average_salary_per_dept.collect():print(f"{dept}: Rp.{avg_salary:,.2f}")

Rata-rata gaji per departemen:
Finance: Rp.940,411.74
IT: Rp.1,129,858.15
HR: Rp.743,853.56
Sales: Rp.792,957.86
R&D: Rp.800,377.16
Marketing: Rp.769,936.15
Operations: Rp.754,626.25


In [13]:
# 4. Jabatan apa yang memiliki rata-rata gaji tertinggi?
job_salary = rdd.map(lambda x: (x['Job_Title'], (float(x['Salary_INR']), 1)))
job_salary_sum = job_salary.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
# Hitung rata-rata
job_avg_salary = job_salary_sum.mapValues(lambda x: x[0] / x[1])
top_job = job_avg_salary.takeOrdered(1, key=lambda x: -x[1])
print("Jabatan dengan rata-rata gaji tertinggi:")
for job, avg_salary in top_job:
    print(f"{job}: Rp {avg_salary:,.2f}")

Jabatan dengan rata-rata gaji tertinggi:
IT Manager: Rp 2,098,155.78


In [14]:
# 5.Berapa rata-rata gaji di berbagai Departemen berdasarkan Jabatan?
dept_job_salary = rdd.map(lambda x: ((x['Department'], x['Job_Title']), (float(x['Salary_INR']), 1)))
dept_job_sum = dept_job_salary.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
# Hitung rata -rata
dept_job_avg = dept_job_sum.mapValues(lambda x: x[0] / x[1])
# Ambil semua hasil
results = dept_job_avg.collect()
print("Rata-rata gaji di berbagai departemen berdasarkan jabatan:")
for (dept, job), avg_salary in results:
    print(f"{dept} - {job}: Rp {avg_salary:,.2f}")

Rata-rata gaji di berbagai departemen berdasarkan jabatan:
IT - DevOps Engineer: Rp 799,949.18
R&D - Research Scientist: Rp 801,314.88
IT - Software Engineer: Rp 1,199,260.84
HR - Talent Acquisition Specialist: Rp 801,422.24
Operations - Supply Chain Manager: Rp 798,168.55
Marketing - Marketing Executive: Rp 798,780.40
IT - CTO: Rp 801,402.75
Sales - Business Development Manager: Rp 1,252,016.23
Operations - Logistics Coordinator: Rp 649,631.73
IT - IT Manager: Rp 2,098,155.78
Marketing - SEO Specialist: Rp 700,456.34
Sales - Sales Executive: Rp 650,237.75
R&D - Product Developer: Rp 798,652.26
Finance - CFO: Rp 795,015.87
Operations - Operations Executive: Rp 800,350.92
Operations - Operations Director: Rp 798,298.09
Marketing - Brand Manager: Rp 803,127.79
Sales - Account Manager: Rp 799,373.73
IT - Data Analyst: Rp 800,996.38
Finance - Finance Manager: Rp 1,743,241.53
Marketing - Content Strategist: Rp 800,760.03
HR - HR Manager: Rp 1,252,401.91
Finance - Accountant: Rp 650,076.48
R

In [15]:
# 6. Hitung jumlah karyawan yang Resigned & Terminated per departemen
dept_status_count = rdd.map(lambda x: ((x['Department'], x['Status']), 1))
dept_status_sum = dept_status_count.reduceByKey(lambda a, b: a + b)

# Ambil semua hasil
results = dept_status_sum.collect()
print("Jumlah karyawan yang mengundurkan diri (Resigned) dan dipecat (Terminated) di setiap departemen:")
for (dept, status), count in results:
    if status in ["Resigned", "Terminated"]:
        print(f"{dept} - {status}: {count}")

Jumlah karyawan yang mengundurkan diri (Resigned) dan dipecat (Terminated) di setiap departemen:
IT - Resigned: 119852
Sales - Terminated: 20214
HR - Resigned: 31736
R&D - Terminated: 4998
Finance - Resigned: 40238
Operations - Terminated: 14884
Marketing - Terminated: 12044
Marketing - Resigned: 47793
Operations - Resigned: 59397
Finance - Terminated: 9988
IT - Terminated: 29881
Sales - Resigned: 79725
R&D - Resigned: 19919
HR - Terminated: 7861


In [16]:
# 7. Variasi gaji berdasarkan tahun pengalaman
exp_salary = rdd.map(lambda x: (x['Experience_Years'], (float(x['Salary_INR']), 1)))
exp_salary_sum = exp_salary.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
# Hitung rata-rata gaji
exp_salary_avg = exp_salary_sum.mapValues(lambda x: x[0] / x[1])
results = exp_salary_avg.collect()
print("Rata-rata gaji berdasarkan pengalaman kerja:")
for exp, avg_salary in sorted(results, key=lambda x: x[0]):  
    print(f"{exp} tahun pengalaman: Rp {avg_salary:,.2f}")


Rata-rata gaji berdasarkan pengalaman kerja:
0 tahun pengalaman: Rp 896,737.45
1 tahun pengalaman: Rp 895,903.76
2 tahun pengalaman: Rp 896,755.65
3 tahun pengalaman: Rp 896,861.25
4 tahun pengalaman: Rp 897,944.57
5 tahun pengalaman: Rp 896,484.08
6 tahun pengalaman: Rp 896,012.63
7 tahun pengalaman: Rp 895,722.67
8 tahun pengalaman: Rp 897,148.36
9 tahun pengalaman: Rp 898,482.94
10 tahun pengalaman: Rp 895,662.03
11 tahun pengalaman: Rp 901,452.75
12 tahun pengalaman: Rp 896,432.93
13 tahun pengalaman: Rp 898,790.20
14 tahun pengalaman: Rp 895,610.79
15 tahun pengalaman: Rp 895,647.40


In [17]:
# 8. Berapa rata-rata peringkat kinerja berdasarkan departemen?
dept_perf = rdd.map(lambda x: (x['Department'], (float(x['Performance_Rating']), 1)))
dept_perf_sum = dept_perf.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
# Hitung rata-rata rating
dept_perf_avg = dept_perf_sum.mapValues(lambda x: x[0] / x[1])
results = dept_perf_avg.collect()
print("Rata-rata peringkat kinerja berdasarkan departemen:")
for dept, avg_rating in sorted(results, key=lambda x: x[0]):
    print(f"{dept}: {avg_rating:.2f}")

Rata-rata peringkat kinerja berdasarkan departemen:
Finance: 3.00
HR: 3.00
IT: 3.00
Marketing: 3.00
Operations: 3.00
R&D: 3.00
Sales: 3.01


In [22]:
# 9.Negara mana yang memiliki konsentrasi karyawan tertinggi?
country_count = rdd.map(lambda x: (x['Location'], 1))
country_sum = country_count.reduceByKey(lambda a, b: a + b)
# Urutkan dari jumlah terbanyak
top_country = country_sum.takeOrdered(10, key=lambda x: -x[1])
# Tampilkan hasil
print("Negara dengan konsentrasi karyawan tertinggi:")
for country, count in top_country:
    print(f"{country}: {count} karyawan")

Negara dengan konsentrasi karyawan tertinggi:
Lake Michael, Congo: 20 karyawan
New Christopher, Congo: 19 karyawan
East Michael, Congo: 17 karyawan
Lake Michael, Bulgaria: 16 karyawan
West Michael, Uzbekistan: 16 karyawan
West Michael, Sao Tome and Principe: 16 karyawan
East David, Congo: 16 karyawan
South Michael, British Indian Ocean Territory (Chagos Archipelago): 16 karyawan
South Michael, Malta: 15 karyawan
Lake David, Korea: 15 karyawan


In [23]:
# 10. Apakah ada korelasi antara peringkat kinerja dan gaji?
from pyspark.mllib.stat import Statistics
perf_salary_rdd = rdd.map(lambda x: (float(x['Performance_Rating']), float(x['Salary_INR'])))
# Hitung korelasi Pearson
rdd_corr = Statistics.corr(perf_salary_rdd.map(lambda x: x[0]), perf_salary_rdd.map(lambda x: x[1]), method="pearson")
print(f"Korelasi antara rating dan gaji: {rdd_corr:.2f}")

Korelasi antara rating dan gaji: -0.00


In [33]:
# 11. Bagaimana jumlah perekrutan berubah dari waktu ke waktu (per tahun)?
import matplotlib.pyplot as plt

hire_year_rdd = rdd.filter(lambda x: x['Hire_Date'] is not None) \
                   .map(lambda x: (x['Hire_Date'].year, 1))  # pakai .year
# Hitung jumlah per tahun
hire_year_count = hire_year_rdd.reduceByKey(lambda a, b: a + b)
# Ambil semua hasil dan urutkan 
hire_year_list = sorted(hire_year_count.collect(), key=lambda x: x[0])  # ascending
# Tampilkan hasil
for year, count in hire_year_list:
    print(f"{year}: {count} karyawan")

2010: 15520 karyawan
2011: 40089 karyawan
2012: 39765 karyawan
2013: 39988 karyawan
2014: 40202 karyawan
2015: 85984 karyawan
2016: 160249 karyawan
2017: 160363 karyawan
2018: 159658 karyawan
2019: 160202 karyawan
2020: 175460 karyawan
2021: 199366 karyawan
2022: 201373 karyawan
2023: 198982 karyawan
2024: 200001 karyawan
2025: 122798 karyawan


In [34]:
# 11. Bandingkan gaji karyawan Remote vs. On-site, apakah ada perbedaan yang signifikan?
mode_salary_rdd = rdd.map(lambda x: (x['Work_Mode'], (float(x['Salary_INR']), 1)))
mode_salary_sum = mode_salary_rdd.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
# Hitung rata-rata gaji
mode_salary_avg = mode_salary_sum.mapValues(lambda x: x[0]/x[1])
# Tampilkan hasil
for mode, avg_salary in mode_salary_avg.collect():
    print(f"{mode}: Rp {avg_salary:,.2f}")

On-site: Rp 896,835.95
Remote: Rp 896,965.33


In [40]:
# 12. Temukan 10 karyawan dengan gaji tertinggi di setiap departemen.
dept_salary_rdd = rdd.map(lambda x: (x['Department'], (x['Full_Name'], float(x['Salary_INR']))))
dept_grouped = dept_salary_rdd.groupByKey()
# Ambil top 10 gaji tertinggi di tiap departemen
top10_per_dept = dept_grouped.mapValues(lambda vals: sorted(vals, key=lambda x: x[1], reverse=True)[:10])
# Tampilkan hasil
for dept, top_employees in top10_per_dept.collect():
    print(f"\nDepartemen: {dept}")
    for name, salary in top_employees:
        print(f" {name}: Rp {salary:,.2f}")


Departemen: Finance
 Christopher Sloan: Rp 2,499,958.00
 Todd Rodgers: Rp 2,499,929.00
 Angela Payne: Rp 2,499,925.00
 Nina Lara: Rp 2,499,813.00
 Brittany Thompson: Rp 2,499,786.00
 Larry Wilson: Rp 2,499,751.00
 Alexis Schroeder: Rp 2,499,732.00
 Sarah Jones: Rp 2,499,674.00
 Jose Anderson: Rp 2,499,629.00
 Jennifer Dominguez: Rp 2,499,601.00

Departemen: IT
 Kathryn Owens: Rp 2,999,976.00
 Robert Bowman: Rp 2,999,973.00
 Christina Delgado: Rp 2,999,944.00
 Donald Cohen: Rp 2,999,906.00
 Brandon Rodriguez: Rp 2,999,889.00
 Dr. David Mitchell: Rp 2,999,881.00
 Cassandra Morales: Rp 2,999,831.00
 Debra Rivera: Rp 2,999,811.00
 Douglas Mann: Rp 2,999,797.00
 Jennifer Reynolds: Rp 2,999,751.00

Departemen: HR
 Ethan Jones: Rp 1,799,839.00
 Austin Hall: Rp 1,799,791.00
 Daniel Wilson: Rp 1,799,769.00
 Amanda Everett: Rp 1,799,759.00
 Michelle Snyder: Rp 1,799,728.00
 Andre Velasquez: Rp 1,799,705.00
 Carrie Davis: Rp 1,799,625.00
 Gregory Pearson: Rp 1,799,598.00
 Dr. Shawn Gibson: Rp 1,

In [41]:
# 13. Identifikasi departemen dengan tingkat attrition (persentase pengunduran diri) tertinggi.
dept_status_rdd = rdd.map(lambda x: ((x['Department'], x['Status']), 1))
dept_status_count = dept_status_rdd.reduceByKey(lambda a, b: a + b)
dept_count_rdd = dept_status_count.map(lambda x: (x[0][0], (x[0][1], x[1])))
# Gabungkan semua status per departemen
from collections import defaultdict
def combine_statuses(iterable):
    total = 0
    resigned = 0
    for status, count in iterable:
        total += count
        if status.lower() in ["resigned", "mengundurkan diri"]:
            resigned += count
    return (resigned, total)
dept_grouped = dept_count_rdd.groupByKey().mapValues(combine_statuses)
# Hitung attrition rate
dept_attrition = dept_grouped.mapValues(lambda x: x[0]/x[1]*100)
# Urutkan descending dan ambil departemen dengan attrition tertinggi
top_attrition = dept_attrition.takeOrdered(1, key=lambda x: -x[1])
# Tampilkan hasil
for dept, rate in top_attrition:
    print(f"Departemen dengan attrition tertinggi: {dept} ({rate:.2f}%)")

Departemen dengan attrition tertinggi: Finance (20.13%)
